# mmBERT 체크포인트 앙상블 추가 실험

기존 `studychat_mmbert_hierarchical_v1`의 세 체크포인트를 같은 비율로 평균합니다. 데이터셋 수정과 재학습은 하지 않습니다.
**A100/H100 권장, T4는 느릴 수 있습니다.** 고용량 RAM 런타임에서 순서대로 실행하세요.
각 체크포인트는 한 번씩 순차적으로 로드하며 추론 배치를 1/1로 고정해 T4 메모리 사용량을 줄입니다.
T4에서 8,192토큰 단일 구간도 OOM이면 A100/H100이 필요합니다. OOM 이후에는 런타임을 다시 시작하세요.

검증셋에서만 판단 기준값을 선택합니다. 기존 단일 모델의 검증 점수보다 앙상블 점수가 높을 때만 마지막 테스트 셀을 실행합니다.
기존 테스트 결과는 선택에 사용하지 않습니다. 점수는 Qwen 자동 검증 라벨과의 일치이며 사람 정답 성능은 아닙니다.
GPU에서 이 노트북의 전체 실행은 아직 검증하지 못했습니다.


## 1. 기존 실행 결과 연결
버전 오류가 나면 2번 셀로 설치하고 런타임을 다시 시작하세요.

In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/ThinkGuard_NLP/studychat_mmbert_hierarchical_v1')
CONFIG_PATH = RUN_DIR / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
if Path(config['run_dir']).resolve() != RUN_DIR.resolve():
    raise ValueError('config.json의 run_dir가 기존 실험 폴더와 다릅니다')
# 추론 배치만 줄입니다. 모델 가중치, 토큰 캐시, 실험 identity는 그대로입니다.
config['window_batch'] = 1
config['record_batch'] = 1
for package, version in config['versions'].items():
    actual = importlib.metadata.version(package)
    if actual != version:
        raise RuntimeError(f'{package} 버전 불일치: 현재 {actual}, 기존 {version}. 다음 설치 셀 실행 후 런타임을 다시 시작하세요.')
CODE = Path('/content/thinkguard_mmbert_ensemble_code')
CODE.mkdir(exist_ok=True)
SOURCES = {'mmtrain/__init__.py': '"""Hierarchical mmBERT training on full source conversations and two weak labels."""\n', 'mmtrain/checkpoint.py': '"""Atomic Drive checkpoints with optimizer and RNG state, only at optimizer boundaries."""\n\nimport os\nfrom pathlib import Path\n\nimport torch\n\n\ndef save(path, state):\n    path = Path(path)\n    temporary = path.with_suffix(\'.tmp\')\n    with temporary.open(\'wb\') as stream:\n        torch.save(state,stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n\n\ndef rng_state():\n    return {\'cpu\':torch.get_rng_state(), \'cuda\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef restore_rng(state):\n    torch.set_rng_state(state[\'cpu\'])\n    if state[\'cuda\'] and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(state[\'cuda\'])\n\n\ndef load(path, identity):\n    value = torch.load(path,map_location=\'cpu\',weights_only=True)\n    if value[\'identity\'] != identity:\n        raise ValueError(\'Checkpoint config/model/data/code identity differs\')\n    return value\n', 'mmtrain/chunks.py': '"""Tokenize all source turns; overlapping context, disjoint ownership for pooling."""\n\n\ndef special_wrapper(tokenizer):\n    probe = \'A short tokenizer probe.\'\n    plain = tokenizer.encode(probe, add_special_tokens=False)\n    wrapped = tokenizer(probe, add_special_tokens=True, return_special_tokens_mask=True)\n    ids, masks = wrapped[\'input_ids\'], wrapped[\'special_tokens_mask\']\n    indexes = [i for i,m in enumerate(masks) if not m]\n    if not indexes or indexes != list(range(indexes[0],indexes[-1]+1)):\n        raise ValueError(\'Unsupported noncontiguous special-token template\')\n    if ids[indexes[0]:indexes[-1]+1] != plain:\n        raise ValueError(\'Special-token template changed input content\')\n    return ids[:indexes[0]], ids[indexes[-1]+1:]\n\n\ndef make_chunks(source, tokenizer, length=4096, overlap=256):\n    if not 16 <= length <= 8192:\n        raise ValueError(\'Chunk length must be between 16 and 8192\')\n    prefix, suffix = special_wrapper(tokenizer)\n    capacity = length-len(prefix)-len(suffix)\n    if not 0 <= overlap < capacity:\n        raise ValueError(\'Overlap must be smaller than content capacity\')\n    tokens, student, current = [], [], []\n    turns = [*enumerate(source[\'history\']), (-1, {\'role\':\'student\',\'content\':source[\'message\']})]\n    for index, turn in turns:\n        if turn[\'role\'] not in (\'student\',\'assistant\') or not isinstance(turn[\'content\'],str):\n            raise ValueError(\'Invalid source role/content\')\n        marker = f\'\\n[TURN={index}; ROLE={turn["role"]}; CURRENT={index == -1}]\\n\'\n        meta = tokenizer.encode(marker, add_special_tokens=False, truncation=False, verbose=False)\n        text = tokenizer.encode(turn[\'content\'], add_special_tokens=False, truncation=False, verbose=False)\n        tokens.extend(meta+text)\n        student.extend([0]*len(meta)+[int(turn[\'role\']==\'student\')]*len(text))\n        current.extend([0]*len(meta)+[int(index==-1)]*len(text))\n    if not tokens:\n        raise ValueError(\'Empty serialized conversation\')\n    chunks, start, owned = [], 0, 0\n    while start < len(tokens):\n        end = min(start+capacity,len(tokens))\n        skip = 0 if not chunks else overlap\n        own = [0]*skip+[1]*(end-start-skip)\n        def mask(values,start=start,end=end,own=own):\n            return [0]*len(prefix)+[v*o for v,o in zip(values[start:end],own,strict=True)]+[0]*len(suffix)\n        chunks.append({\'input_ids\':prefix+tokens[start:end]+suffix,\n                       \'student_mask\':mask(student), \'current_mask\':mask(current),\n                       \'owner_mask\':[0]*len(prefix)+own+[0]*len(suffix),\n                       \'start\':start,\'end\':end})\n        owned += sum(own)\n        if end == len(tokens):\n            break\n        start = end-overlap\n    if owned != len(tokens) or sum(sum(c[\'current_mask\']) for c in chunks) != sum(current):\n        raise ValueError(\'Token coverage or overlap ownership failure\')\n    return chunks, {\'tokens\':len(tokens),\'single_input_tokens\':len(tokens)+len(prefix)+len(suffix),\n                    \'chunks\':len(chunks), \'current_tokens\':sum(current),\n                    \'student_tokens\':sum(student),\'encoded_tokens\':sum(len(c[\'input_ids\']) for c in chunks)}\n', 'mmtrain/data.py': '"""Frozen CSV IDs, provenance and group leakage checks; labels never enter text."""\n\nimport csv\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\n\nLABELS = (\'requests_complete_answer\',\'has_self_attempt_evidence\')\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef stable(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n\n\ndef load_sources(directory):\n    directory = Path(directory)\n    labels, splits, students, chats = {}, {}, {}, {}\n    for split in (\'train\',\'valid\',\'test\'):\n        with (directory/f\'{split}_ok.csv\').open(encoding=\'utf-8-sig\',newline=\'\') as stream:\n            rows = list(csv.DictReader(stream))\n        splits[split] = []\n        for row in rows:\n            uid = row[\'dataset\']+\':\'+row[\'source_id\']\n            if row[\'dataset\'] != \'studychat\' or uid in labels or row[\'status\'] != \'ok\':\n                raise ValueError(\'Duplicate ID or non-ok StudyChat row\')\n            if any(row[k] not in (\'true\',\'false\') for k in (*LABELS,\'is_learning_context\')):\n                raise ValueError(\'Null or malformed label in _ok CSV\')\n            for key, owners in ((\'student_id\',students),(\'conversation_id\',chats)):\n                if not row[key] or owners.setdefault(row[key],split) != split:\n                    raise ValueError(\'Student/conversation overlaps splits\')\n            labels[uid] = row\n            splits[split].append(uid)\n    sources = {}\n    with gzip.open(directory/\'inputs.jsonl.gz\',\'rt\',encoding=\'utf-8\') as stream:\n        for line in stream:\n            row = json.loads(line)\n            uid = row[\'dataset\']+\':\'+row[\'source_id\']\n            if uid not in labels:\n                continue\n            if uid in sources or row.get(\'input_verified\') is not True:\n                raise ValueError(\'Duplicate or unverified original\')\n            if stable({k:row[k] for k in (\'message\',\'history\')}) != row[\'input_sha256\']:\n                raise ValueError(\'Original text hash mismatch\')\n            if any(row[k] != labels[uid][k] for k in (\'student_id\',\'conversation_id\')):\n                raise ValueError(\'CSV/source identity mismatch\')\n            sources[uid] = row\n    if set(sources) != set(labels):\n        raise ValueError(\'CSV ID missing from original source\')\n    # Fixed label-independent policy. Never move IDs or inspect test predictions.\n    seen, excluded, kept = {}, [], {}\n    for split in (\'test\',\'valid\',\'train\'):\n        kept[split] = []\n        for uid in splits[split]:\n            digest = sources[uid][\'input_sha256\']\n            if digest in seen:\n                excluded.append({\'id\':uid,\'split\':split,\'duplicate_of\':seen[digest], \'reason\':\'exact_input_duplicate\'})\n            else:\n                seen[digest] = uid\n                kept[split].append(uid)\n    if any(not kept[split] for split in (\'train\',\'valid\',\'test\')):\n        raise ValueError(\'A split is empty after duplicate removal\')\n    return sources, labels, kept, excluded\n\n\ndef load_cache(config):\n    import torch\n    for name,digest in config[\'inputs\'].items():\n        if sha(Path(config[\'data_dir\'])/name) != digest:\n            raise ValueError(\'Frozen source snapshot changed\')\n    manifest = json.loads((Path(config[\'run_dir\'])/\'token_cache_manifest.json\').read_text(encoding=\'utf-8\'))\n    path = Path(config[\'run_dir\'])/\'tokenized.pt\'\n    if manifest[\'identity\'] != config[\'identity\'] or sha(path) != manifest[\'sha256\']:\n        raise ValueError(\'Token cache identity/content changed\')\n    return torch.load(path,map_location=\'cpu\',weights_only=True)\n', 'mmtrain/evaluate.py': '"""Final held-out evaluation with fixed validation thresholds and per-row predictions."""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .checkpoint import load\nfrom .data import load_cache, sha\nfrom .metrics import evaluate\nfrom .runtime import initialize, predict\n\n\ndef run(config):\n    root = Path(config[\'run_dir\'])\n    committed = load(root/\'latest.pt\',config[\'identity\'])\n    best_name = committed[\'best_file\']\n    if best_name is None:\n        raise ValueError(\'No completed validation checkpoint; finish training first\')\n    del committed\n    best_path, report_path = root/best_name,root/\'test_report.json\'\n    digest = sha(best_path)\n    if report_path.exists():\n        report = json.loads(report_path.read_text(encoding=\'utf-8\'))\n        if report[\'checkpoint_sha256\'] != digest or report[\'identity\'] != config[\'identity\']:\n            raise ValueError(\'Existing test report is for a different checkpoint/config\')\n        print(\'Reusing completed test report\',json.dumps(report,indent=2))\n        return\n    state = load(best_path,config[\'identity\'])\n    cache = load_cache(config)\n    model,_ = initialize(config)\n    model.load_state_dict(state[\'model\'])\n    records, ids = cache[\'records\'],cache[\'splits\'][\'test\']\n    target, probabilities, seconds = predict(model,records,ids,config[\'record_batch\'])\n    report = evaluate(target,probabilities,state[\'thresholds\'])\n    prevalence = np.mean([records[u][\'labels\'] for u in cache[\'splits\'][\'train\']],axis=0)\n    report[\'majority_baseline\'] = evaluate(target,np.tile(prevalence,(len(ids),1)),[.5,.5])\n    report.update(identity=config[\'identity\'],checkpoint_sha256=digest,predict_seconds=seconds,\n                  rows_per_second=len(ids)/max(seconds,1e-9),valid_report=state[\'valid_report\'])\n    temp = root/\'test_predictions.tmp\'\n    with temp.open(\'w\',encoding=\'utf-8\') as stream:\n        for uid,truth,prob in zip(ids,target,probabilities,strict=True):\n            stream.write(json.dumps({\'id\':uid,\'input_sha256\':records[uid][\'input_sha256\'],\n                \'target\':truth.tolist(),\'probabilities\':prob.tolist(),\n                \'prediction\':(prob>=np.asarray(state[\'thresholds\'])).tolist()})+\'\\n\')\n    temp.replace(root/\'test_predictions.jsonl\')\n    temporary = report_path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    temporary.replace(report_path)\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':\n    import sys\n    run(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\')))\n', 'mmtrain/metrics.py': '"""Binary-class macro-F1 for each task; thresholds selected on validation only."""\n\nimport numpy as np\n\nfrom .data import LABELS\n\n\ndef binary_metrics(target, prediction):\n    target, prediction = np.asarray(target,dtype=bool),np.asarray(prediction,dtype=bool)\n    tp,tn = int(np.sum(target&prediction)),int(np.sum(~target&~prediction))\n    fp,fn = int(np.sum(~target&prediction)),int(np.sum(target&~prediction))\n    def ratio(a,b):\n        return a/b if b else 0.\n    positive_f1, negative_f1 = ratio(2*tp,2*tp+fp+fn),ratio(2*tn,2*tn+fp+fn)\n    return {\'macro_f1\':(positive_f1+negative_f1)/2,\'positive_f1\':positive_f1,\n            \'precision\':ratio(tp,tp+fp),\'recall\':ratio(tp,tp+fn),\'fpr\':fp/(fp+tn) if fp+tn else None,\n            \'tp\':tp,\'tn\':tn,\'fp\':fp,\'fn\':fn,\'positive_support\':tp+fn,\'negative_support\':tn+fp}\n\n\ndef choose_thresholds(target, probabilities):\n    target, probabilities = np.asarray(target),np.asarray(probabilities)\n    result = []\n    for i in range(2):\n        options = [(binary_metrics(target[:,i],probabilities[:,i]>=t)[\'macro_f1\'],-abs(t-.5),-t,t)\n                   for t in np.linspace(.1,.9,17)]\n        result.append(float(max(options)[-1]))\n    return result\n\n\ndef evaluate(target, probabilities, thresholds):\n    target, probabilities = np.asarray(target),np.asarray(probabilities)\n    if target.shape != probabilities.shape or target.ndim!=2 or target.shape[1]!=2 or len(target)==0:\n        raise ValueError(\'Expected nonempty two-label arrays\')\n    prediction = probabilities>=np.asarray(thresholds)\n    labels = {key:binary_metrics(target[:,i],prediction[:,i]) for i,key in enumerate(LABELS)}\n    return {\'labels\':labels,\'mean_label_macro_f1\':float(np.mean([v[\'macro_f1\'] for v in labels.values()])),\n            \'exact_match\':float(np.mean(np.all(target.astype(bool)==prediction,axis=1))),\n            \'thresholds\':list(thresholds),\'rows\':len(target),\n            \'interpretation\':\'Agreement with automatically verified Qwen labels, not human-gold accuracy.\'}\n', 'mmtrain/model.py': '"""Full-token window encoder plus current-conditioned context and student pooling."""\n\nimport torch\nfrom torch import nn\nfrom torch.utils.checkpoint import checkpoint\n\n\ndef pad_windows(chunks, pad_id, device):\n    length = max(len(c[\'input_ids\']) for c in chunks)\n    result = {}\n    for key in (\'input_ids\',\'student_mask\',\'current_mask\',\'owner_mask\'):\n        fill = pad_id if key==\'input_ids\' else 0\n        result[key] = torch.tensor([c[key]+[fill]*(length-len(c[key])) for c in chunks],device=device)\n    result[\'attention_mask\'] = torch.tensor([[1]*len(c[\'input_ids\'])+[0]*(length-len(c[\'input_ids\']))\n                                            for c in chunks],device=device)\n    return result\n\n\nclass DialogueClassifier(nn.Module):\n    def __init__(self, encoder, hidden, pad_id, window_batch=2, dropout=.1):\n        super().__init__()\n        self.encoder, self.pad_id, self.window_batch = encoder, pad_id, window_batch\n        self.position = nn.Linear(3,hidden)\n        self.context_key = nn.Linear(hidden,hidden)\n        self.student_key = nn.Linear(hidden,hidden)\n        self.query_key = nn.Linear(hidden,hidden)\n        self.context_score = nn.Linear(hidden,1)\n        self.student_score = nn.Linear(hidden,1)\n        self.heads = nn.ModuleList([nn.Sequential(nn.LayerNorm(hidden*4),nn.Dropout(dropout),\n                                    nn.Linear(hidden*4,hidden),nn.GELU(),nn.Linear(hidden,1)) for _ in range(2)])\n\n    def encode_pool(self, ids, attention, student, current, owner):\n        hidden = self.encoder(input_ids=ids,attention_mask=attention).last_hidden_state.float()\n        def pool(mask):\n            mask = mask.unsqueeze(-1).to(hidden.dtype)\n            return (hidden*mask).sum(1)/mask.sum(1).clamp(min=1)\n        return torch.stack([pool(owner),pool(student),pool(current)],dim=1)\n\n    def encode_chunks(self, chunks):\n        device = next(self.parameters()).device\n        features = []\n        for start in range(0,len(chunks),self.window_batch):\n            batch = pad_windows(chunks[start:start+self.window_batch],self.pad_id,device)\n            args = [batch[k] for k in (\'input_ids\',\'attention_mask\',\'student_mask\',\'current_mask\',\'owner_mask\')]\n            if self.training and torch.is_grad_enabled():\n                features.append(checkpoint(self.encode_pool,*args,use_reentrant=False,preserve_rng_state=True))\n            else:\n                features.append(self.encode_pool(*args))\n        return torch.cat(features).float()\n\n    def classify(self, chunks, features, return_attention=False):\n        device = features.device\n        current_counts = torch.tensor([sum(c[\'current_mask\']) for c in chunks],device=device,dtype=torch.float32)\n        student_counts = torch.tensor([sum(c[\'student_mask\']) for c in chunks],device=device,dtype=torch.float32)\n        owner_counts = torch.tensor([sum(c[\'owner_mask\']) for c in chunks],device=device,dtype=torch.float32)\n        query = (features[:,2]*current_counts[:,None]).sum(0)/current_counts.sum().clamp(min=1)\n        positions = torch.stack([torch.arange(len(chunks),device=device)/max(len(chunks)-1,1),\n                                current_counts/owner_counts.clamp(min=1),\n                                student_counts/owner_counts.clamp(min=1)],dim=1)\n        position = self.position(positions)\n        def attend(values, key, score, available):\n            scores = score(torch.tanh(key(values)+self.query_key(query)[None]+position)).squeeze(-1)\n            weights = torch.softmax(scores.masked_fill(~available,-1e9),dim=0)\n            weights = weights*available\n            weights = weights/weights.sum().clamp(min=1e-12)\n            return (weights[:,None]*values).sum(0),weights\n        context, ca = attend(features[:,0],self.context_key,self.context_score,owner_counts>0)\n        student, sa = attend(features[:,1],self.student_key,self.student_score,student_counts>0)\n        logits = []\n        for head, pooled in zip(self.heads,(context,student),strict=True):\n            combined = torch.cat([query,pooled,query*pooled,torch.abs(query-pooled)])\n            logits.append(head(combined).squeeze(-1))\n        logits = torch.stack(logits)\n        return (logits,{\'context\':ca,\'student\':sa}) if return_attention else logits\n\n    def forward(self, chunks, return_attention=False):\n        return self.classify(chunks,self.encode_chunks(chunks),return_attention)\n\n    def forward_many(self, bags):\n        flat = [chunk for bag in bags for chunk in bag]\n        features = self.encode_chunks(flat)\n        outputs, start = [], 0\n        for bag in bags:\n            outputs.append(self.classify(bag,features[start:start+len(bag)]))\n            start += len(bag)\n        return torch.stack(outputs)\n\n\ndef build_model(config):\n    from transformers import AutoConfig, AutoModel\n    hf = AutoConfig.from_pretrained(config[\'model_id\'],revision=config[\'revision\'])\n    if hf.max_position_embeddings < config[\'chunk_length\']:\n        raise ValueError(\'Chunk exceeds pretrained context length\')\n    hf.reference_compile = False\n    encoder = AutoModel.from_pretrained(config[\'model_id\'],revision=config[\'revision\'],config=hf,\n                                        attn_implementation=\'sdpa\')\n    encoder.float()\n    return DialogueClassifier(encoder,hf.hidden_size,config[\'pad_id\'],config[\'window_batch\'])\n', 'mmtrain/prepare.py': '"""Lossless token cache plus coverage, length, duplicate and class reports."""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .chunks import make_chunks\nfrom .data import LABELS, load_cache, load_sources, sha\n\n\ndef prepare(config):\n    from transformers import AutoTokenizer\n    root = Path(config[\'run_dir\'])\n    if (root/\'token_cache_manifest.json\').exists():\n        cache = load_cache(config)\n        print(\'Reusing token cache:\',{k:len(v) for k,v in cache[\'splits\'].items()},flush=True)\n        return\n    for name,digest in config[\'inputs\'].items():\n        if sha(Path(config[\'data_dir\'])/name) != digest:\n            raise ValueError(\'Input changed after configuration was frozen\')\n    sources, labels, splits, excluded = load_sources(config[\'data_dir\'])\n    tokenizer = AutoTokenizer.from_pretrained(config[\'model_id\'],revision=config[\'revision\'])\n    records, split_report = {}, {}\n    for split, ids in splits.items():\n        lengths, windows = [], []\n        for i,uid in enumerate(ids):\n            chunks, stats = make_chunks(sources[uid],tokenizer,config[\'chunk_length\'],config[\'overlap\'])\n            row = {k:sources[uid][k] for k in (\'dataset\',\'source_id\',\'student_id\',\'conversation_id\',\'input_sha256\')}\n            row.update(chunks=chunks,labels=[float(labels[uid][k]==\'true\') for k in LABELS],stats=stats)\n            records[uid] = row\n            lengths.append(stats[\'single_input_tokens\'])\n            windows.append(stats[\'chunks\'])\n            if (i+1)%250==0:\n                print(split,\'tokenized\',i+1,\'/\',len(ids),flush=True)\n        split_report[split] = {\'rows\':len(ids),\'tokens_percentiles\':dict(zip((\'p50\',\'p90\',\'p95\',\'p99\',\'max\'),\n                                  np.percentile(lengths,[50,90,95,99,100]).astype(int).tolist(),strict=True)),\n                              \'over_8192\':sum(n>8192 for n in lengths),\'max_chunks\':max(windows),\n                              \'total_chunks\':sum(windows),\n                              \'positives\':{key:int(sum(records[uid][\'labels\'][j] for uid in ids))\n                                           for j,key in enumerate(LABELS)}}\n    cache = {\'records\':records,\'splits\':splits,\'excluded\':excluded,\'identity\':config[\'identity\']}\n    temp = root/\'tokenized.tmp\'\n    torch.save(cache,temp)\n    temp.replace(root/\'tokenized.pt\')\n    manifest = {\'identity\':config[\'identity\'],\'sha256\':sha(root/\'tokenized.pt\'),\'splits\':split_report,\n                \'excluded_exact_duplicates\':excluded,\'original_csv_rows\':len(sources),\n                \'kept_rows\':len(records),\'all_tokens_processed\':True,\'max_chunks_cap\':None}\n    (root/\'token_cache_manifest.json\').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(json.dumps({k:v for k,v in manifest.items() if k!=\'excluded_exact_duplicates\'},ensure_ascii=False,indent=2))\n\n\nif __name__ == \'__main__\':\n    import sys\n    prepare(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\')))\n', 'mmtrain/runtime.py': '"""GPU initialization, equal-record loss and prediction independent of test selection."""\n\nimport contextlib\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom .model import build_model\n\n\ndef initialize(config):\n    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\n        raise RuntimeError(\'A100/H100 BF16 GPU required\')\n    torch.manual_seed(config[\'seed\'])\n    torch.cuda.manual_seed_all(config[\'seed\'])\n    print(\'GPU:\',torch.cuda.get_device_name(0),\'window batch:\',config[\'window_batch\'],\n          \'record batch:\',config[\'record_batch\'],flush=True)\n    model = build_model(config).cuda()\n    encoder_ids = {id(p) for p in model.encoder.parameters()}\n    optimizer = torch.optim.AdamW([\n        {\'params\':[p for p in model.parameters() if id(p) in encoder_ids], \'lr\':config[\'encoder_lr\']},\n        {\'params\':[p for p in model.parameters() if id(p) not in encoder_ids], \'lr\':config[\'head_lr\']}],\n        weight_decay=.01)\n    return model,optimizer\n\n\ndef amp(model):\n    return torch.autocast(\'cuda\',dtype=torch.bfloat16) if next(model.parameters()).is_cuda else contextlib.nullcontext()\n\n\ndef loss_group(model, records, ids, record_batch, pos_weight=None):\n    """One equally weighted loss per conversation, regardless of number of windows."""\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    device = next(model.parameters()).device\n    total = 0.\n    for start in range(0,len(ids),record_batch):\n        current = ids[start:start+record_batch]\n        target = torch.tensor([records[uid][\'labels\'] for uid in current],device=device)\n        with amp(model):\n            logits = model.forward_many([records[uid][\'chunks\'] for uid in current])\n            loss = criterion(logits,target)*(len(current)/len(ids))\n        if not torch.isfinite(loss):\n            raise ValueError(\'Nonfinite training loss\')\n        loss.backward()\n        total += float(loss.detach())\n    return total\n\n\ndef predict(model, records, ids, record_batch):\n    model.eval()\n    probabilities, target = [], []\n    if next(model.parameters()).is_cuda:\n        torch.cuda.synchronize()\n    started = time.monotonic()\n    with torch.inference_mode():\n        for start in range(0,len(ids),record_batch):\n            current = ids[start:start+record_batch]\n            with amp(model):\n                logits = model.forward_many([records[uid][\'chunks\'] for uid in current])\n            probabilities.extend(torch.sigmoid(logits.float()).cpu().tolist())\n            target.extend(records[uid][\'labels\'] for uid in current)\n    if next(model.parameters()).is_cuda:\n        torch.cuda.synchronize()\n    return np.asarray(target),np.asarray(probabilities),time.monotonic()-started\n', 'mmtrain/train.py': '"""Full encoder fine-tuning, validation-only selection, exact optimizer-boundary resume."""\n\nimport argparse\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .checkpoint import load, restore_rng, rng_state, save\nfrom .data import load_cache\nfrom .metrics import choose_thresholds, evaluate\nfrom .runtime import initialize, loss_group, predict\n\n\ndef epoch_ids(ids, records, epoch, seed):\n    generator = torch.Generator().manual_seed(seed+epoch)\n    ordered = sorted(ids,key=lambda uid:records[uid][\'stats\'][\'chunks\'])\n    buckets = [ordered[i:i+64] for i in range(0,len(ordered),64)]\n    result = []\n    for index in torch.randperm(len(buckets),generator=generator).tolist():\n        bucket = buckets[index]\n        result.extend(bucket[j] for j in torch.randperm(len(bucket),generator=generator).tolist())\n    return result\n\n\ndef train(config, smoke=False):\n    cache = load_cache(config)\n    records, splits = cache[\'records\'],cache[\'splits\']\n    root = Path(config[\'run_dir\'])\n    model, optimizer = initialize(config)\n    train_ids = splits[\'train\']\n    prevalence = np.mean([records[uid][\'labels\'] for uid in train_ids],axis=0)\n    weights = np.clip((1-prevalence)/np.maximum(prevalence,1/len(train_ids)),.5,5)\n    pos_weight = torch.tensor(weights,device=\'cuda\',dtype=torch.float32) if config[\'class_weights\'] else None\n    if smoke:\n        chosen = sorted(train_ids,key=lambda uid:records[uid][\'stats\'][\'chunks\'])\n        chosen = list(dict.fromkeys(chosen[:2]+chosen[-1:]))\n        started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss = loss_group(model,records,chosen,config[\'record_batch\'],pos_weight)\n        if not any(p.grad is not None and torch.count_nonzero(p.grad).item() for p in model.encoder.parameters()):\n            raise ValueError(\'Encoder gradients missing\')\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        torch.cuda.synchronize()\n        report = {\'loss\':loss,\'sample_rows\':len(chosen),\n                  \'max_sample_chunks\':max(records[u][\'stats\'][\'chunks\'] for u in chosen),\n                  \'seconds\':time.monotonic()-started,\'peak_vram_gib\':torch.cuda.max_memory_allocated()/2**30,\n                  \'note\':\'Smoke includes the longest train conversation; weights are NOT used for full training.\'}\n        (root/\'smoke_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        print(json.dumps(report,indent=2))\n        return\n    steps_per_epoch = math.ceil(len(train_ids)/config[\'effective_batch\'])\n    total = steps_per_epoch*config[\'epochs\']\n    warmup = max(1,int(total*.1))\n    def multiplier(step):\n        return step/warmup if step<warmup else .5*(1+math.cos(math.pi*(step-warmup)/max(1,total-warmup)))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,multiplier)\n    epoch,offset,step,best,bad_epochs,best_file = 0,0,0,-1.,0,None\n    latest = root/\'latest.pt\'\n    if latest.exists():\n        state = load(latest,config[\'identity\'])\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        scheduler.load_state_dict(state[\'scheduler\'])\n        epoch,offset,step,best,bad_epochs = [state[k] for k in (\'epoch\',\'offset\',\'step\',\'best\',\'bad_epochs\')]\n        best_file = state[\'best_file\']\n        restore_rng(state[\'rng\'])\n        del state\n        print(\'Resuming optimizer step\',step,\'epoch\',epoch,\'next row offset\',offset,flush=True)\n    def checkpoint(next_epoch, next_offset):\n        save(latest,{\'identity\':config[\'identity\'],\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\n                     \'scheduler\':scheduler.state_dict(),\'rng\':rng_state(),\'epoch\':next_epoch,\'offset\':next_offset,\n                     \'step\':step,\'best\':best,\'bad_epochs\':bad_epochs,\'best_file\':best_file})\n        print(\'Drive checkpoint committed:\',step,flush=True)\n    if bad_epochs >= config[\'patience\']:\n        print(\'Early stopping already satisfied; using best.pt\',flush=True)\n        return\n    while epoch<config[\'epochs\']:\n        ordered = epoch_ids(train_ids,records,epoch,config[\'seed\'])\n        model.train()\n        started = time.monotonic()\n        for start in range(offset,len(ordered),config[\'effective_batch\']):\n            group = ordered[start:start+config[\'effective_batch\']]\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_group(model,records,group,config[\'record_batch\'],pos_weight)\n            norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                raise ValueError(\'Nonfinite gradient norm\')\n            optimizer.step()\n            scheduler.step()\n            step += 1\n            if step%10==0:\n                print(\'epoch\',epoch+1,\'step\',step,\'rows\',start+len(group),\'/\',len(ordered),\n                      \'loss\',round(loss,4),\'elapsed sec\',round(time.monotonic()-started,1),flush=True)\n            if step%config[\'save_steps\']==0 and start+len(group)<len(ordered):\n                checkpoint(epoch,start+len(group))\n        target, probabilities, seconds = predict(model,records,splits[\'valid\'],config[\'record_batch\'])\n        thresholds = choose_thresholds(target,probabilities)\n        report = evaluate(target,probabilities,thresholds)\n        report.update(epoch=epoch+1,optimizer_step=step,predict_seconds=seconds,\n                      peak_vram_gib=torch.cuda.max_memory_allocated()/2**30)\n        score = report[\'mean_label_macro_f1\']\n        if score>best:\n            best,bad_epochs = score,0\n            best_file = f\'best_step_{step}.pt\'\n            save(root/best_file,{\'identity\':config[\'identity\'],\'model\':model.state_dict(),\n                                 \'thresholds\':thresholds,\'valid_report\':report})\n            (root/\'best_valid_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        else:\n            bad_epochs += 1\n        with (root/\'training_log.jsonl\').open(\'a\',encoding=\'utf-8\') as stream:\n            stream.write(json.dumps(report)+\'\\n\')\n        print(json.dumps(report,indent=2),flush=True)\n        epoch,offset = epoch+1,0\n        checkpoint(epoch,0)\n        if bad_epochs>=config[\'patience\']:\n            print(\'Early stopping on VALID only\',flush=True)\n            break\n    print(\'Training complete; committed best model and thresholds:\',root/best_file,flush=True)\n\n\nif __name__==\'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    parser.add_argument(\'--smoke\',action=\'store_true\')\n    args = parser.parse_args()\n    train(json.loads(Path(args.config).read_text(encoding=\'utf-8\')),args.smoke)\n'}
for relative, source in SOURCES.items():
    path = CODE / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from mmtrain.data import load_cache, sha
from mmtrain.checkpoint import load
from mmtrain.metrics import choose_thresholds, evaluate
from mmtrain.runtime import initialize, predict
import numpy as np
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 필요합니다')
print('GPU:', torch.cuda.get_device_name(0), '추론 window/record batch: 1/1')

CHECKPOINTS = ['best_step_541.pt', 'best_step_1082.pt', 'best_step_1623.pt']
for name in CHECKPOINTS:
    if not (RUN_DIR / name).is_file():
        raise FileNotFoundError(RUN_DIR / name)
cache = load_cache(config)  # 원본 해시·토큰 캐시 identity 확인
records, splits = cache['records'], cache['splits']
baseline = json.loads((RUN_DIR / 'best_valid_report.json').read_text(encoding='utf-8'))
if baseline['optimizer_step'] != 1623:
    raise ValueError('기존 best가 step 1623이 아닙니다. 체크포인트 목록을 재검토하세요.')
print('valid/test rows:', len(splits['valid']), len(splits['test']))
print('기존 valid 점수:', baseline['mean_label_macro_f1'])


## 2. 필요할 때만 환경 설치

In [ ]:
# setup 셀에서 버전 불일치가 발생했을 때만 실행하세요.
import json, subprocess, sys
from pathlib import Path
run_dir = Path('/content/drive/MyDrive/ThinkGuard_NLP/studychat_mmbert_hierarchical_v1')
versions = json.loads((run_dir / 'environment_versions.json').read_text(encoding='utf-8'))
requirements = [f'{name}=={version}' for name, version in versions.items()]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])
print('설치 완료. Colab 런타임을 다시 시작한 뒤 setup 셀부터 실행하세요.')


## 3. 검증셋 비교
세 체크포인트를 순서대로 추론합니다. 검증 점수가 기존보다 엄격히 높아야 테스트가 열립니다.

In [ ]:
def checkpoint_predictions(name, ids):
    state = load(RUN_DIR / name, config['identity'])
    model = optimizer = None
    try:
        model, optimizer = initialize(config)
        model.load_state_dict(state['model'])
        del state
        target, probabilities, seconds = predict(model, records, ids, config['record_batch'])
        return target, probabilities, seconds
    finally:
        if model is not None:
            del model
        if optimizer is not None:
            del optimizer
        gc.collect()
        torch.cuda.empty_cache()

valid_target = None
valid_probabilities = []
valid_seconds = {}
for name in CHECKPOINTS:
    target, probabilities, seconds = checkpoint_predictions(name, splits['valid'])
    if valid_target is None:
        valid_target = target
    elif not np.array_equal(valid_target, target):
        raise ValueError('검증셋 라벨·순서가 체크포인트 사이에 다릅니다')
    valid_probabilities.append(probabilities)
    valid_seconds[name] = seconds
    print(name, 'valid seconds:', round(seconds, 1))

# 기존 단일 모델 결과가 그대로 재현되는지 확인합니다.
single_state = load(RUN_DIR / CHECKPOINTS[-1], config['identity'])
single_valid = evaluate(valid_target, valid_probabilities[-1], single_state['thresholds'])
if abs(single_valid['mean_label_macro_f1'] - baseline['mean_label_macro_f1']) > 1e-9:
    raise ValueError('기존 단일 모델 valid 점수가 재현되지 않습니다')
del single_state

ensemble_valid_prob = np.mean(np.stack(valid_probabilities), axis=0)
ensemble_thresholds = choose_thresholds(valid_target, ensemble_valid_prob)
ensemble_valid = evaluate(valid_target, ensemble_valid_prob, ensemble_thresholds)
IMPROVED = ensemble_valid['mean_label_macro_f1'] > single_valid['mean_label_macro_f1'] + 1e-12
print('기존 단일 모델:', json.dumps(single_valid, ensure_ascii=False, indent=2))
print('3개 평균 앙상블:', json.dumps(ensemble_valid, ensure_ascii=False, indent=2))
print('valid 향상:', IMPROVED)

OUT_DIR = RUN_DIR / 'checkpoint_ensemble_v1'
OUT_DIR.mkdir(exist_ok=True)
validation_result = {
    'checkpoints': CHECKPOINTS,
    'checkpoint_sha256': {name: sha(RUN_DIR / name) for name in CHECKPOINTS},
    'identity': config['identity'],
    'selection_rule': 'equal probability mean of all three checkpoints; validation threshold grid 0.1..0.9 step 0.05; test only if valid score strictly improves',
    'single_valid': single_valid,
    'ensemble_valid': ensemble_valid,
    'improved': IMPROVED,
    'valid_seconds': valid_seconds,
}
(OUT_DIR / 'validation_report.json').write_text(json.dumps(validation_result, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장:', OUT_DIR / 'validation_report.json')


## 4. 조건부 최종 테스트
3번 셀의 결과를 확인한 뒤 실행하세요. 원래 단일 모델 테스트와 같은 고정 테스트셋을 사용합니다.

In [ ]:
if not IMPROVED:
    print('검증셋에서 향상되지 않아 테스트 평가를 건너뜁니다.')
else:
    current = json.loads((OUT_DIR / 'validation_report.json').read_text(encoding='utf-8'))
    if current['identity'] != config['identity'] or current['checkpoint_sha256'] != {name: sha(RUN_DIR / name) for name in CHECKPOINTS}:
        raise ValueError('검증 이후 체크포인트 또는 설정이 바뀌었습니다')
    report_path = OUT_DIR / 'test_report.json'
    if report_path.exists():
        print('기존 앙상블 테스트 보고서 재사용:', report_path.read_text(encoding='utf-8'))
    else:
        test_target = None
        test_probabilities = []
        test_seconds = {}
        for name in CHECKPOINTS:
            target, probabilities, seconds = checkpoint_predictions(name, splits['test'])
            if test_target is None:
                test_target = target
            elif not np.array_equal(test_target, target):
                raise ValueError('테스트셋 라벨·순서가 체크포인트 사이에 다릅니다')
            test_probabilities.append(probabilities)
            test_seconds[name] = seconds
            print(name, 'test seconds:', round(seconds, 1))
        test_prob = np.mean(np.stack(test_probabilities), axis=0)
        ensemble_test = evaluate(test_target, test_prob, ensemble_thresholds)
        original_test = json.loads((RUN_DIR / 'test_report.json').read_text(encoding='utf-8'))
        ensemble_test.update(
            identity=config['identity'],
            checkpoint_sha256=current['checkpoint_sha256'],
            validation_report='validation_report.json',
            original_single_test_mean_macro_f1=original_test['mean_label_macro_f1'],
            delta_vs_single_test=ensemble_test['mean_label_macro_f1'] - original_test['mean_label_macro_f1'],
            test_seconds=test_seconds,
        )
        report_path.write_text(json.dumps(ensemble_test, ensure_ascii=False, indent=2), encoding='utf-8')
        print(json.dumps(ensemble_test, ensure_ascii=False, indent=2))
